# Hybrid Quantum-Classical Drug Discovery — Colab Demo

Based on the project abstract: a hybrid framework that uses **QAOA** to optimize
drug–protein binding configurations and a **quantum-inspired affinity adjustment**
to support ranking of candidate molecules, combined with classical docking scores.

This notebook uses **synthetic data** to illustrate the pipeline end-to-end, and
serves a **Gradio** frontend with a public shareable link.

Run each cell in order (Runtime → Run all also works).

## 1. Install dependencies

In [ ]:
!pip install -q qiskit qiskit-algorithms qiskit-optimization gradio

## 2. Generate synthetic candidate molecules

Stands in for the "large numbers of compounds" step in the abstract. Each
candidate gets a synthetic classical docking score plus a few molecular
descriptors (molecular weight, logP, rotatable bonds).

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

def generate_candidates(n=6):
    names = [f"Compound-{i+1}" for i in range(n)]
    docking_score = np.random.uniform(-9, -5, n)      # kcal/mol; more negative = stronger predicted binding
    molecular_weight = np.random.uniform(180, 500, n)
    logP = np.random.uniform(-1, 5, n)
    rotatable_bonds = np.random.randint(1, 10, n)
    return pd.DataFrame({
        "name": names,
        "docking_score": docking_score,
        "molecular_weight": molecular_weight,
        "logP": logP,
        "rotatable_bonds": rotatable_bonds,
    })

generate_candidates(6)

## 3. QAOA-based candidate selection

Formulates "pick the best `budget` candidates" as a constrained optimization
problem (a small QUBO) and solves it with QAOA. This mirrors the abstract's use
of QAOA to optimize drug–protein binding configurations — here simplified to
selecting the most promising subset of candidates under a budget constraint.

In [ ]:
from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import StatevectorSampler

def run_qaoa_selection(df, budget):
    n = len(df)
    benefit = -df["docking_score"].values  # flip sign so "higher benefit" = stronger binding

    qp = QuadraticProgram("drug_selection")
    for i in range(n):
        qp.binary_var(name=f"x{i}")

    linear = {f"x{i}": float(benefit[i]) for i in range(n)}
    qp.maximize(linear=linear)

    qp.linear_constraint(
        linear={f"x{i}": 1 for i in range(n)},
        sense="==",
        rhs=budget,
        name="budget_constraint",
    )

    qaoa = QAOA(sampler=StatevectorSampler(), optimizer=COBYLA(maxiter=100), reps=2)
    optimizer = MinimumEigenOptimizer(qaoa)
    result = optimizer.solve(qp)

    selection = [bool(round(v)) for v in result.x]
    return selection, result


## 4. Quantum-inspired affinity adjustment

A small parameterized single-qubit circuit whose expectation value depends on
a molecular descriptor (logP), used to adjust the classical docking score. This
is a simplified stand-in for the abstract's "Quantum Machine Learning" step —
without real training data, it illustrates *where* a trained QML model would
plug into the pipeline rather than performing real learned prediction.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit import Parameter

def quantum_affinity_adjustment(row):
    theta = Parameter("theta")
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.ry(theta, 0)

    scaled_logP = (row["logP"] + 1) / 6 * np.pi  # map logP range into [0, pi]

    observable = SparsePauliOp("Z")
    estimator = StatevectorEstimator()
    job = estimator.run([(qc, observable, [scaled_logP])])
    exp_val = float(job.result()[0].data.evs)

    return exp_val * 0.5


## 5. Combine and rank candidates

Combines the QAOA selection with the quantum-adjusted docking score into a
final ranked table, mirroring the abstract's "combine quantum predictions with
classical results to improve candidate evaluation" step.

In [ ]:
def run_pipeline(n_candidates, budget):
    df = generate_candidates(n_candidates)
    selection, qaoa_result = run_qaoa_selection(df, budget)
    df["qaoa_selected"] = selection

    df["quantum_adjustment"] = df.apply(quantum_affinity_adjustment, axis=1)
    df["predicted_binding_affinity"] = df["docking_score"] - df["quantum_adjustment"]

    ranked_df = df.sort_values("predicted_binding_affinity").reset_index(drop=True)
    ranked_df["rank"] = ranked_df.index + 1
    return ranked_df

## 6. Gradio frontend

Sliders control how many synthetic candidates to generate and how many QAOA
should select. Running the cell launches a public link (`share=True`) you can
open in a browser tab.

In [ ]:
import gradio as gr
import matplotlib.pyplot as plt
import tempfile, os

CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Serif:wght@500;600&display=swap');

:root {
    --ink: #2B2620;
    --paper: #EFEAE1;
    --surface: #FAF8F4;
    --moss: #5F6F44;
    --ochre: #C08A3E;
    --hairline: #DDD5C7;
}

.gradio-container { background: var(--paper) !important; }
input[type="range"] { accent-color: var(--moss); }

/* ---- Hero ---- */
.hero {
    background: var(--ink);
    border-radius: 4px;
    padding: 40px 36px 34px 36px;
    margin-bottom: 28px;
}
.hero .eyebrow {
    color: var(--ochre);
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.8em;
    letter-spacing: 0.02em;
    margin-bottom: 10px;
}
.hero h1 {
    font-family: 'IBM Plex Serif', serif;
    font-weight: 600;
    font-size: 2.1em;
    color: var(--surface) !important;
    margin: 0 0 10px 0;
    line-height: 1.25;
    max-width: 14ch;
}
.hero p {
    color: #C9C1B3 !important;
    font-size: 1em;
    max-width: 62ch;
    line-height: 1.55;
    margin: 0;
}

/* ---- Step timeline ---- */
.timeline {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 0;
    margin: 8px 0 28px 0;
    position: relative;
}
.timeline::before {
    content: "";
    position: absolute;
    top: 19px;
    left: 6%;
    right: 6%;
    height: 1px;
    background: var(--hairline);
}
.step { position: relative; padding: 0 18px 0 0; }
.step-num {
    display: flex;
    align-items: center;
    justify-content: center;
    width: 38px; height: 38px;
    border-radius: 50%;
    border: 1.5px solid var(--moss);
    color: var(--moss);
    background: var(--paper);
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.95em;
    position: relative;
    z-index: 1;
    margin-bottom: 14px;
}
.step h3 {
    font-family: 'IBM Plex Serif', serif;
    font-weight: 600;
    font-size: 1.02em;
    color: var(--ink);
    margin: 0 0 6px 0;
}
.step p { font-size: 0.88em; color: #5C5646; line-height: 1.5; margin: 0; }

/* ---- About panel ---- */
.panel {
    border-left: 3px solid var(--ochre);
    background: var(--surface);
    padding: 18px 22px;
    font-size: 0.93em;
    line-height: 1.6;
    color: #45402F;
}
.panel b { color: var(--ink); }

/* ---- Section labels ---- */
.section-label {
    font-family: 'IBM Plex Mono', monospace;
    font-size: 0.82em;
    color: #7A7461;
    margin: 4px 0 14px 0;
}

/* ---- Run button: small, quiet, not a banner ---- */
.run-row { margin-top: 6px; margin-bottom: 18px; }
.run-btn button {
    width: auto !important;
    min-width: 0 !important;
    background: var(--moss) !important;
    border: 1px solid var(--moss) !important;
    color: var(--surface) !important;
    font-weight: 500 !important;
    font-size: 0.92em !important;
    padding: 8px 22px !important;
    box-shadow: none !important;
}
.run-btn button:hover { background: #4E5C38 !important; }

/* ---- Result summary ---- */
.result-summary {
    font-family: 'IBM Plex Mono', monospace !important;
    font-size: 0.88em !important;
    color: var(--ink) !important;
    background: var(--surface);
    border: 1px solid var(--hairline);
    padding: 10px 14px;
    border-radius: 4px;
}

/* ---- Tabs ---- */
.tabs .tab-nav button.selected {
    color: var(--moss) !important;
    border-color: var(--moss) !important;
}
"""

STEP_HTML = """
<div class="timeline">
  <div class="step">
    <div class="step-num">01</div>
    <h3>Generate candidates</h3>
    <p>Synthetic candidate molecules with a classical docking score plus
    descriptors (molecular weight, logP, rotatable bonds), standing in for a
    compound library screened against a target protein.</p>
  </div>
  <div class="step">
    <div class="step-num">02</div>
    <h3>QAOA selection</h3>
    <p>Choosing the best subset under a budget is framed as a QUBO problem and
    solved with QAOA — a quantum algorithm built for combinatorial
    optimization.</p>
  </div>
  <div class="step">
    <div class="step-num">03</div>
    <h3>Quantum adjustment</h3>
    <p>A parameterized single-qubit circuit encodes each molecule's logP as a
    rotation angle, producing an expectation value that adjusts the docking
    score — a placeholder for a trained quantum ML model.</p>
  </div>
  <div class="step">
    <div class="step-num">04</div>
    <h3>Rank &amp; recommend</h3>
    <p>Docking score and quantum adjustment combine into a predicted binding
    affinity; candidates are ranked, with QAOA-selected ones marked.</p>
  </div>
</div>
"""

ABOUT_HTML = """
<div class="panel">
<b>Project</b> — Hybrid Quantum-Classical Drug Discovery Framework<br><br>
Uses QAOA to optimize drug–protein binding configurations and quantum machine
learning to predict binding affinity and molecular properties. Quantum
predictions are combined with classical ML and docking results to rank
candidate drug molecules, which are then validated classically before lab
testing.<br><br>
This demo uses synthetic data to illustrate the pipeline architecture end-to-end.
</div>
"""

def gradio_pipeline(n_candidates, budget):
    budget = min(budget, n_candidates)
    ranked_df = run_pipeline(n_candidates, budget)

    plt.rcParams["font.family"] = "monospace"
    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ["#5F6F44" if sel else "#C9C2B2" for sel in ranked_df["qaoa_selected"]]
    ax.barh(ranked_df["name"], ranked_df["predicted_binding_affinity"], color=colors)
    ax.set_xlabel("Predicted Binding Affinity (lower = stronger)", fontsize=9)
    ax.set_title("Ranked Drug Candidates  ·  moss green = QAOA-selected", fontsize=10, color="#2B2620")
    ax.set_facecolor("#FAF8F4")
    fig.patch.set_facecolor("#FAF8F4")
    ax.spines[["top", "right"]].set_visible(False)
    ax.invert_yaxis()
    plt.tight_layout()

    display_df = ranked_df[["rank", "name", "docking_score", "quantum_adjustment",
                             "predicted_binding_affinity", "qaoa_selected"]]

    csv_path = os.path.join(tempfile.gettempdir(), "ranked_candidates.csv")
    display_df.to_csv(csv_path, index=False)

    summary = (
        f"{int(ranked_df['qaoa_selected'].sum())} of {n_candidates} candidates selected by QAOA  ·  "
        f"top pick: {ranked_df.iloc[0]['name']}  "
        f"(predicted affinity {ranked_df.iloc[0]['predicted_binding_affinity']:.2f})"
    )

    return display_df, fig, csv_path, summary

theme = gr.themes.Base(
    primary_hue=gr.themes.colors.stone,
    neutral_hue=gr.themes.colors.stone,
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "monospace"],
)

with gr.Blocks(title="Hybrid Quantum-Classical Drug Discovery", css=CUSTOM_CSS, theme=theme) as demo:

    gr.HTML("""
    <div class="hero">
      <div class="eyebrow">qiskit &middot; qaoa &middot; synthetic demo</div>
      <h1>Hybrid Quantum-Classical Drug Discovery</h1>
      <p>QAOA-based candidate selection combined with a quantum-assisted
      binding-affinity adjustment, ranking synthetic drug candidates against
      a target protein.</p>
    </div>
    """)

    with gr.Tabs():
        with gr.Tab("How it works"):
            gr.HTML('<div class="section-label">the pipeline, step by step</div>')
            gr.HTML(STEP_HTML)
            gr.HTML('<div class="section-label" style="margin-top:6px;">about this project</div>')
            gr.HTML(ABOUT_HTML)

        with gr.Tab("Run the demo"):
            gr.HTML('<div class="section-label">configure and run the pipeline</div>')
            with gr.Row():
                n_slider = gr.Slider(4, 8, value=6, step=1, label="Number of candidate molecules")
                budget_slider = gr.Slider(1, 6, value=3, step=1, label="Number to select (QAOA budget)")
            with gr.Row(elem_classes="run-row"):
                with gr.Column(scale=0, min_width=180):
                    run_btn = gr.Button("Run pipeline", variant="primary", elem_classes="run-btn")
                with gr.Column():
                    pass
            summary_md = gr.HTML()
            output_table = gr.Dataframe(label="Ranked candidates")
            output_plot = gr.Plot(label="Binding affinity ranking")
            download_btn = gr.File(label="Download ranked results (CSV)")

            def wrapped(n, b):
                df, fig, csv_path, summary = gradio_pipeline(n, b)
                return df, fig, csv_path, f'<div class="result-summary">{summary}</div>'

            run_btn.click(
                wrapped,
                inputs=[n_slider, budget_slider],
                outputs=[output_table, output_plot, download_btn, summary_md],
            )

demo.launch(share=True)
